This is an example of obtain 1) DOC injection fluxes and 2) [DOC]-time to drive ATS 2D transect simualations

- Input
    - 1
    - 2
- Output
    - 1
    - 2

**Version History**

**2025/10/17**
- add concentration unit convertion from per bulk volume to per water volumn

**2025/10/14**
- add `config.json`

**2025/10/8**
- add hillslope average plot over time
- add NH4+, NO3- extraction

**2025/8/14**
- output DOC fluxes h5 files
    - without fdom and soil moisture scaling
    - fdom and soil moisture scaling are handled by final h5 generation for 2D hillslope model
- output soil moisture h5 files
    - used by possible scaling
- output LIT and SOM pools concententration vertically resolved
    - for potential estimation of DOC concentration

In [ ]:
%load_ext autoreload
%autoreload 2

# Parameters and data sources

In [ ]:
# Parameters cell
import json
with open('/global/cfs/cdirs/m1800/xiaoyi/Code/watershed-workflow-2dhillslope/notebooks/config.json', 'r') as f:
    config = json.load(f)
watershed_name = config['watershed_name']
hucs           = [config['hucs']]
site_name      = config['site_name']

# simulation control
start_year_spinup         = config['start_year_spinup']
end_year_spinup           = config['end_year_spinup']
nyears_steadystate_spinup = config['nyears_steadystate_spinup']
nyears_cyclic_spinup      = config['nyears_cyclic_spinup']
start_year_transient      = config['start_year_transient']
end_year_transient        = config['end_year_transient']

In [ ]:
soil_thickness_median = 1.52
rho_m = 55000. # moles/m^3, water molar density
outputs={}

In [ ]:
import os, sys
import glob
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import h5py as h5
from tqdm import tqdm, trange
from scipy.io import loadmat
import math

In [ ]:
import h5py
from pyproj import Transformer
from scipy.spatial import cKDTree

In [ ]:
# from makegif import make_gif
import cartopy
from herbie import Herbie #[Yi] Herbie is used by Zhi to process HRRR data?
from herbie.toolbox import EasyMap, pc
naches_wbd = np.loadtxt('/pscratch/sd/x/xiao284/elmbyhuilin/WBD_xyz/naches_wbd.xyz')
oakcreek_wbd = np.loadtxt('/pscratch/sd/x/xiao284/elmbyhuilin/WBD_xyz/oakcreek_wbd.xyz')

generate_animation = False

In [ ]:
import shapely
from shapely.geometry import Point,Polygon,mapping
import geopandas as gpd

import watershed_workflow
import watershed_workflow.source_list
import watershed_workflow.ui
import watershed_workflow.colors
import watershed_workflow.condition
import watershed_workflow.mesh
import watershed_workflow.split_hucs
import watershed_workflow.soil_properties
import watershed_workflow.daymet
import watershed_workflow.utils
import watershed_workflow.regions

# Prepare watershed shape and hillslope shape

In [ ]:
# load hillslope geometry from mat file generated in "1-full_workflow_OakCreek.ipynb"
meshsize_nx=100

m2_mat_filename =  f'./data-processed/{site_name}/m2_{site_name}_nx{meshsize_nx}.mat'
loaded_data = loadmat(m2_mat_filename)
#dzs_soil  = loaded_data['dzs_soil'].flatten()
#dzs_geo   = loaded_data['dzs_geo'].flatten()
#m2_coords = loaded_data['m2_coords']
loaded_gdf_dict = loaded_data['gdf_data']
gdf_reloaded = pd.DataFrame({
    'lon': loaded_gdf_dict['lon'][0, 0].flatten(),
    'lat': loaded_gdf_dict['lat'][0, 0].flatten(),
    'h_distance': loaded_gdf_dict['h_distance'][0, 0].flatten(),
    'elevation': loaded_gdf_dict['elevation'][0, 0].flatten()
})
geometry = [Point(xy) for xy in zip(gdf_reloaded['lon'], gdf_reloaded['lat'])]
hillslope_gdf = gpd.GeoDataFrame(gdf_reloaded, geometry=geometry)

# create hillslope polygon and shape object
xsec_plg = Polygon([hillslope_gdf.geometry[i] for i in range(hillslope_gdf.shape[0])])
xsec_plg_dict = {"type": "Feature", "id":0, "properties":{}, "geometry": mapping(xsec_plg)}
xsec_plg_dict_shply = watershed_workflow.utils.create_shply(xsec_plg_dict)

proj_daymet = "+proj=lcc +lat_1=25 +lat_2=60 +lat_0=42.5 +lon_0=-100 +x_0=0 +y_0=0 +datum=WGS84" # daymet crs
proj_wgs84  = "epsg:4326" # latlon
crs_daymet  = watershed_workflow.crs.from_string(proj_daymet)
crs_wgs84   = watershed_workflow.crs.from_string(proj_wgs84)

# convert to destination crs crs_wgs84
reproj_bnd = watershed_workflow.warp.shape(xsec_plg_dict, crs_daymet, crs_wgs84)
reproj_bnd_shply = watershed_workflow.utils.create_shply(reproj_bnd)

In [ ]:
gdf_reloaded

In [ ]:
def read_daymet_h5(filename):
    data = {}
    with h5py.File(filename, 'r') as f:
        for k, v in f.items():
            try:
                data[k] = v[:]
            except TypeError:
                data_t = {}
                for tk, tv in v.items():
                    data_t[tk] = tv[:]
                data[k] = data_t
    return data

In [ ]:
def write_daymet_h5(filename, data):
    with h5py.File(filename, 'w') as f:
        for k, v in data.items():
            try:
                f.create_dataset(k, data=v)
            except TypeError:
                g = f.create_group(k)
                for tk, tv in v.items():
                    g.create_dataset(tk, data=tv)

In [ ]:
def check_keys(data):
    keys = data.keys()
    for key in keys:
        print(key, type(data[key]))
        if isinstance(data[key], dict):
            _keys = data[key].keys()
            for _key in _keys:
                if int(_key) > 5:
                    break
                print('\t', _key, ': type is', type(data[key][_key]))

In [ ]:
# print("mpl - figure.facecolor:", mpl.rcParams["figure.facecolor"])
# print("mpl - axes.facecolor:", mpl.rcParams["axes.facecolor"])
# print("mpl - savefig.facecolor:", mpl.rcParams["savefig.facecolor"]) # in the IPython notebook with an inline backend, where the "saved" version of the figure you see below the cell is not controlled by the figure parameter, but by the savefig paramter.
# print("plt - axes.facecolor:", plt.rcParams['axes.facecolor'])
# print("plt - grid.color:", plt.rcParams['grid.color'])
# print("Active style sheets:", plt.style.available)

# mpl.rcParams.update(mpl.rcParamsDefault)
# plt.style.use('default')
# mpl.rcParams["savefig.facecolor"] = "white"

# # Verify the reset worked:
# print("After reset - axes.facecolor:", plt.rcParams['axes.facecolor'])
# print("After reset - grid.color:", plt.rcParams['grid.color'])

In [ ]:
#plt.style.use('ggplot')
plt.rcParams['figure.dpi'] = 100
# plt.rcParams['figure.figsize'] = (10,4)
plt.rcParams['lines.linewidth'] = 1
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10

# Load ELM results

In [ ]:
run = '2024-07-14-142626' # '2024-07-08-110357' '2024-07-14-142626' ignition:'2024-08-27-132724' post-fire:'2024-08-27-150810'
                          # fire: '2024-11-20-210107' no fire: '2024-11-20-220109'
#path = f'/pscratch/sd/l/lizh142/elm/ELM_MOSART_CONUS.{run}/run/'
#path = f'/compass/ber200003/zhi/elm/ELM_MOSART_CONUS.{run}/run/'
path = f'/pscratch/sd/x/xiao284/elmbyhuilin/elm/ELM_MOSART_CONUS.{run}/run/'
years = np.arange(2020, 2021)

In [ ]:
f = path+f'ELM_MOSART_CONUS.{run}.elm.h0.{years[-1]}-01-01-00000.nc'
# f = path+f'ELM_MOSART_CONUS.{run}.elm.h0.2021-08-05-00000.nc'
data = xr.open_dataset(f)
data

## Get keys and units

In [ ]:
keys, names, units = [], [], []
for key in list(data.keys()):
    try:
        keys.append(key)
    except:
        keys.append('')
    try:
        names.append(data[key].long_name)
    except:
        names.append('')
    try:
        units.append(data[key].units)
    except:
        units.append('')
df = pd.DataFrame(data={'keys': keys, 'names': names, 'units': units})
# df.to_csv('all_ELM_vars.csv')
df

In [ ]:
# search keyword
search_keyword = 'NH4'
idx = []
for i in range(len(df)):
    if search_keyword in df['keys'][i] or search_keyword in df['names'][i] or search_keyword in df['units'][i]:
        idx.append(i)
df.iloc[idx, :]

In [ ]:
# search keyword
search_keyword = 'NO3'
idx = []
for i in range(len(df)):
    if search_keyword in df['keys'][i] or search_keyword in df['names'][i] or search_keyword in df['units'][i]:
        idx.append(i)
df.iloc[idx, :]

In [ ]:
# search keyword
search_keyword = '_vr'
idx = []
for i in range(len(df)):
    if search_keyword in df['keys'][i] or search_keyword in df['names'][i] or search_keyword in df['units'][i]:
        idx.append(i)
df.iloc[idx, :]

In [ ]:
# search keyword
search_keyword = 'infiltration'
idx = []
for i in range(len(df)):
    if search_keyword in df['keys'][i] or search_keyword in df['names'][i] or search_keyword in df['units'][i]:
        idx.append(i)
df.iloc[idx, :]

## Plot ELM grid, watersheds boundary, and hillslope
- plot soil moisture - top layer - with Naches/Oak Creek/hillslope


In [ ]:
lon, lat = data.coords['lon'].values-360, data.coords['lat'].values
X, Y = np.meshgrid(lon, lat)

# Flatten X, Y into points
elm_mesh_points = np.column_stack([X.ravel(), Y.ravel()])
tree = cKDTree(elm_mesh_points)
line_coords = list(reproj_bnd_shply.exterior.coords)

# Find nearest cell for each point along the line
distances, indices_flat = tree.query(line_coords)
indices_flat = np.unique(indices_flat)

# Convert flat indices back to 2D indices
hillslope_elm_lat_indices = indices_flat // X.shape[1]
hillslope_elm_lon_indices = indices_flat % X.shape[1]

print(X.shape)
print([len(lat), len(lon)])
print(hillslope_elm_lat_indices)
print(hillslope_elm_lon_indices)

In [ ]:
fig, [ax1, ax2, ax3] = plt.subplots(1, 3, figsize=(12,4), subplot_kw={'projection': cartopy.crs.LambertConformal()})
i=0 # 1st layer

# raw
EasyMap("10m", ax=ax1, crs=cartopy.crs.LambertConformal()).STATES().OCEAN().LAND().COUNTIES().LAKES()
ax1.set_extent([lon.min(), lon.max(), lat.min(), lat.max()], crs=cartopy.crs.PlateCarree())
ax1.add_feature(cartopy.feature.RIVERS)
art = ax1.pcolormesh(X, Y, data['H2OSOI'].values[60, i, :, :], cmap='Spectral_r', vmin=0, vmax=1, transform=cartopy.crs.PlateCarree())
fig.colorbar(art, label='[mm3/mm3]', shrink=0.5, extend='max', orientation='vertical', pad=0.02)
ax1.set_title(f'layer {i+1}: {str(np.around(data.levgrnd.values[i], 3))} m')
# add specified boundaries
ax1.plot(naches_wbd[:,0]+360, naches_wbd[:,1], 'k--', lw=2, transform=pc)
ax1.plot(oakcreek_wbd[:,0]+360, oakcreek_wbd[:,1], 'k', lw=2, transform=pc)
watershed_workflow.plot.shply(reproj_bnd_shply, crs_wgs84, ax=ax1, color='r')

# zoom in level1
lon_min_zoom = np.min(oakcreek_wbd[:, 0])+360
lon_max_zoom = np.max(oakcreek_wbd[:, 0])+360
lat_min_zoom = np.min(oakcreek_wbd[:, 1])
lat_max_zoom = np.max(oakcreek_wbd[:, 1])

EasyMap("10m", ax=ax2, crs=cartopy.crs.LambertConformal()).STATES().OCEAN().LAND().COUNTIES().LAKES()
ax2.add_feature(cartopy.feature.RIVERS)
art = ax2.pcolormesh(X, Y, data['H2OSOI'].values[60, i, :, :], cmap='Spectral_r', vmin=0, vmax=1, transform=cartopy.crs.PlateCarree())
fig.colorbar(art, label='[mm3/mm3]', shrink=0.5, extend='max', orientation='vertical', pad=0.02)
ax2.set_title(f'layer {i+1}: {str(np.around(data.levgrnd.values[i], 3))} m')
# add specified boundaries
#ax2.plot(naches_wbd[:,0]+360, naches_wbd[:,1], 'k--', lw=2, transform=pc)
ax2.plot(oakcreek_wbd[:,0]+360, oakcreek_wbd[:,1], 'k', lw=2, transform=pc)
watershed_workflow.plot.shply(reproj_bnd_shply, crs_wgs84, ax=ax2, color='r')
ax2.set_extent([lon_min_zoom, lon_max_zoom, lat_min_zoom, lat_max_zoom], crs=cartopy.crs.PlateCarree())

# zoom in level2
lon_min_zoom, lat_min_zoom, lon_max_zoom, lat_max_zoom = reproj_bnd_shply.bounds
lon_min_zoom +=360
lon_max_zoom +=360
buffer = 0.005

#EasyMap("10m", ax=ax3, crs=cartopy.crs.LambertConformal()).STATES().OCEAN().LAND().COUNTIES().LAKES()
#ax3.add_feature(cartopy.feature.RIVERS)
art = ax3.pcolormesh(X, Y, data['H2OSOI'].values[60, i, :, :], cmap='Spectral_r', vmin=0, vmax=1, transform=cartopy.crs.PlateCarree())
fig.colorbar(art, label='[mm3/mm3]', shrink=0.5, extend='max', orientation='vertical', pad=0.02)
ax3.set_title(f'layer {i+1}: {str(np.around(data.levgrnd.values[i], 3))} m')
# add specified boundaries
#ax3.plot(naches_wbd[:,0]+360, naches_wbd[:,1], 'k--', lw=2, transform=pc)
ax3.plot(oakcreek_wbd[:,0]+360, oakcreek_wbd[:,1], 'k', lw=2, transform=pc)
watershed_workflow.plot.shply(reproj_bnd_shply, crs_wgs84, ax=ax3, color='r')
if False: # can make it True after found overlap ELM grids
    ax3.scatter(X[hillslope_elm_lat_indices, hillslope_elm_lon_indices], 
                Y[hillslope_elm_lat_indices, hillslope_elm_lon_indices], 
               s=50, c='blue', marker='x', linewidths=2,
               transform=cartopy.crs.PlateCarree(), 
               label='Nearest cells', zorder=10)
ax3.set_extent([lon_min_zoom - buffer, lon_max_zoom + buffer, lat_min_zoom - buffer, lat_max_zoom + buffer], crs=cartopy.crs.PlateCarree())

plt.show()

# Get ELM for the hillslope site

similar to the `get_Daymet.ipynb`
- first, interpolate data to x y coordinate of 2D hillslope -> `raw_dat_2dtran`
- then, warp `raw_dat_2dtran` to coordinate (0,680)x(0,1) -> `raw_dat_2dtran_warped`
- lastly, write to hdf5 file

In [ ]:
# years of data used to generate the source term and boundary conditions
# noticing the difference to nyears_steadystate_spinup and nyears_cyclic_spinup loaded from config.json
years_spinup    = np.arange(start_year_spinup, end_year_spinup+1)
years_transient = np.arange(start_year_transient, end_year_transient+1)

print(list(years_spinup))
print(list(years_transient))

## Find value at x and y in 2D transect

Use Zhi's strategy in `read_ELM_byZhi.ipynb`

In [ ]:
# 1. read ELM Naches lon and lat
lon, lat = data.coords['lon'].values-360, data.coords['lat'].values
X, Y = np.meshgrid(lon, lat)

# 2. convert lon lot to crs_daymet
lonlat = np.zeros((len(X.flatten()), 2))
lonlat[:, 0], lonlat[:, 1] = X.flatten(),  Y.flatten()
#proj_lcc = "+proj=lcc +lat_1=25 +lat_2=60 +lat_0=42.5 +lon_0=-100 +x_0=0 +y_0=0 +datum=WGS84" # daymet crs
#proj_wgs84 = "epsg:4326" # latlon
lonlat_to_daymet = np.array(Transformer.from_crs(proj_wgs84, proj_daymet).transform(lonlat[:, 1], lonlat[:, 0]))

In [ ]:
# 3. 2d hillslope mesh in daymet crs
fig, axes = plt.subplots(1, 2, figsize=(12,4))
ax = axes.flatten()

ax[0].scatter(lonlat_to_daymet[0], lonlat_to_daymet[1], s=0.5, c='red')
ax[0].scatter(gdf_reloaded['lon'], gdf_reloaded['lat'], s=0.5, c='blue')
ax[0].axis('equal')

ax[1].scatter(lonlat_to_daymet[0], lonlat_to_daymet[1], s=2, c='red')
ax[1].scatter(gdf_reloaded['lon'], gdf_reloaded['lat'], s=2, c='blue')
# Set axis limits based on gdf_reloaded extent with a small buffer
lon_buffer = (gdf_reloaded['lon'].max() - gdf_reloaded['lon'].min()) * 3
lat_buffer = (gdf_reloaded['lat'].max() - gdf_reloaded['lat'].min()) * 10
ax[1].set_xlim(gdf_reloaded['lon'].min() - lon_buffer, 
               gdf_reloaded['lon'].max() + lon_buffer)
ax[1].set_ylim(gdf_reloaded['lat'].min() - lat_buffer, 
               gdf_reloaded['lat'].max() + lat_buffer)
ax[1].set_aspect('equal', adjustable='box')  # Use set_aspect instead of axis('equal')

plt.tight_layout()
plt.show()

In [ ]:
# 4. prepare interpolation
xy_source = np.zeros((len(lonlat_to_daymet[0]), 2))
xy_source[:, 0], xy_source[:, 1] = lonlat_to_daymet[0], lonlat_to_daymet[1]

target_x_h5file = gdf_reloaded['h_distance']
target_y_h5file = np.array([0.0, 1.0])

xy_target = np.zeros((len(gdf_reloaded['lon']) * 2, 2))
xy_target[:, 0] = np.concatenate([gdf_reloaded['lon'], gdf_reloaded['lon']])
xy_target[:, 1] = np.concatenate([gdf_reloaded['lat'], gdf_reloaded['lat']])

In [ ]:
#target_x_h5file # 0-680
#xy_source[:, 0] # easting
#xy_source[:, 1] # northing
#xy_target[:, 0] # easting
#xy_target[:, 1] # northing

In [ ]:
def idw_interpolation(xy_target, xy_source, values, power=2):
    tree = cKDTree(xy_source)
    distances, indices = tree.query(xy_target, k=4)
    weights = 1 / (distances ** power)
    weights /= weights.sum(axis=1, keepdims=True)
    interpolated_values = np.sum(values[indices] * weights, axis=1)
    return interpolated_values

In [ ]:
# 5. find ELM data
pre_fire = True # True False

# _keys = [#'CWDC_TO_LITR2C', 'CWDC_TO_LITR3C', 
#          'LITR1C_TO_SOIL1C', 'LITR2C_TO_SOIL2C', 'LITR3C_TO_SOIL3C', 
#          'SOIL1C_TO_SOIL2C', 'SOIL2C_TO_SOIL3C', 'SOIL3C_TO_SOIL4C']
#          #'HR', 'SMIN_NH4_vr', 'SMIN_NO3_vr']

elm_flux_data_spinup_raw = {
    "DOC production latlon [molC m^-3 s^-1]": {},
    "DOC production [molC m^-3 s^-1]": {},
    "time [s]": (np.arange(len(years_spinup)*365))*86400,
    "x [m]": target_x_h5file,
    "y [m]": target_y_h5file,
}
elm_conc_data_spinup_raw = {
    "ELM NH4+ [mol L^-1]": np.zeros((len(years_spinup)*365, len(data.coords["levgrnd"]))),
    "ELM NO3- [mol L^-1]": np.zeros((len(years_spinup)*365, len(data.coords["levgrnd"]))),
    "ELM water volume per bulk volume [L L^-1]": np.zeros((len(years_spinup)*365, len(data.coords["levgrnd"]))),
    "NH4+ bulk volume basis [molS L^-1]": np.zeros(len(years_spinup)*365),
    "NO3- bulk volume basis [molS L^-1]": np.zeros(len(years_spinup)*365),
    "NH4+ mol water basis [molS molH^-1]": np.zeros(len(years_spinup)*365),
    "NO3- mol water basis [molS molH^-1]": np.zeros(len(years_spinup)*365),
    "time [d]": np.zeros(len(years_spinup)*365),
    "time [s]": (np.arange(len(years_spinup)*365))*86400,
    "DOC bulk volume basis v1 [molS L^-1]": np.zeros(len(years_spinup)*365),
    "DOC mol water basis v1 [molS molH^-1]": np.zeros(len(years_spinup)*365),
    "DOC bulk volume basis v2 [molS L^-1]": np.zeros(len(years_spinup)*365),
    "DOC mol water basis v2 [molS molH^-1]": np.zeros(len(years_spinup)*365)
}
# unit molS/molH is required by ATS xml input
# need unit conversion: from ELM unit gN/m^3 to ATS unit mol/L
elm_carbon_data_spinup_raw = {
    "ELM LITR1C_vr [mol L^-1]": np.zeros((len(years_spinup)*365, len(data.coords["levgrnd"]))),
    "ELM LITR2C_vr [mol L^-1]": np.zeros((len(years_spinup)*365, len(data.coords["levgrnd"]))),
    "ELM LITR3C_vr [mol L^-1]": np.zeros((len(years_spinup)*365, len(data.coords["levgrnd"]))),
    "ELM SOIL1C_vr [mol L^-1]": np.zeros((len(years_spinup)*365, len(data.coords["levgrnd"]))),
    "ELM SOIL2C_vr [mol L^-1]": np.zeros((len(years_spinup)*365, len(data.coords["levgrnd"]))),
    "ELM SOIL3C_vr [mol L^-1]": np.zeros((len(years_spinup)*365, len(data.coords["levgrnd"]))),
    "ELM SOIL4C_vr [mol L^-1]": np.zeros((len(years_spinup)*365, len(data.coords["levgrnd"])))
}

elm_flux_data_transient = {
    "DOC production latlon [molC m^-3 s^-1]": {},
    "DOC production [molC m^-3 s^-1]": {},
    "time [s]": (np.arange(len(years_transient)*365))*86400,
    "x [m]": target_x_h5file,
    "y [m]": target_y_h5file,
}
elm_conc_data_transient = {
    "ELM NH4+ [mol L^-1]": np.zeros((len(years_transient)*365, len(data.coords["levgrnd"]))),
    "ELM NO3- [mol L^-1]": np.zeros((len(years_transient)*365, len(data.coords["levgrnd"]))),
    "ELM water volume per bulk volume [L L^-1]": np.zeros((len(years_transient)*365, len(data.coords["levgrnd"]))),
    "NH4+ bulk volume basis [molS L^-1]": np.zeros(len(years_transient)*365),
    "NO3- bulk volume basis [molS L^-1]": np.zeros(len(years_transient)*365),
    "NH4+ mol water basis [molS molH^-1]": np.zeros(len(years_transient)*365),
    "NO3- mol water basis [molS molH^-1]": np.zeros(len(years_transient)*365),
    "time [d]": np.zeros(len(years_transient)*365),
    "time [s]": (np.arange(len(years_transient)*365))*86400,
    "DOC bulk volume basis v1 [molS L^-1]": np.zeros(len(years_transient)*365),
    "DOC mol water basis v1 [molS molH^-1]": np.zeros(len(years_transient)*365),
    "DOC bulk volume basis v2 [molS L^-1]": np.zeros(len(years_transient)*365),
    "DOC mol water basis v2 [molS molH^-1]": np.zeros(len(years_transient)*365)
}
elm_carbon_data_transient = {
    "ELM LITR1C_vr [mol L^-1]": np.zeros((len(years_transient)*365, len(data.coords["levgrnd"]))),
    "ELM LITR2C_vr [mol L^-1]": np.zeros((len(years_transient)*365, len(data.coords["levgrnd"]))),
    "ELM LITR3C_vr [mol L^-1]": np.zeros((len(years_transient)*365, len(data.coords["levgrnd"]))),
    "ELM SOIL1C_vr [mol L^-1]": np.zeros((len(years_transient)*365, len(data.coords["levgrnd"]))),
    "ELM SOIL2C_vr [mol L^-1]": np.zeros((len(years_transient)*365, len(data.coords["levgrnd"]))),
    "ELM SOIL3C_vr [mol L^-1]": np.zeros((len(years_transient)*365, len(data.coords["levgrnd"]))),
    "ELM SOIL4C_vr [mol L^-1]": np.zeros((len(years_transient)*365, len(data.coords["levgrnd"])))
}
# unit molS/molH is required by ATS xml input
# need unit conversion: from ELM unit gN/m^3 to ATS unit mol/L

# add DOC

### Find soil layers in ELM 
- "soil" layers info. is used to obtain the averaged [NH4+] and [NO3-]
- "soil" layers info. is also used to obtain the overall water content, to convert concentrations per bulk volume to concentrations per water volume or per mole water
- based on soil_thickness_median from 1-main_workflow_OakCreek.NF01.ats1.5.ipynb

In [ ]:
for layer in range(len(data.coords["levgrnd"])):
    tmp_total_thickness = sum(data.coords["levgrnd"][0:layer+1]).item()
    print(tmp_total_thickness)
    if tmp_total_thickness > soil_thickness_median:
        num_soil_layer_thickness_median = layer + 1
        break
soil_layer_thickness_array = (data.coords["levgrnd"][0:num_soil_layer_thickness_median]).values

print(num_soil_layer_thickness_median) # number of layers used to calculate the NH4+ and NO3+
print(soil_layer_thickness_array)
print(sum(soil_layer_thickness_array))

### Find values for DOC injection and transport BC

In [ ]:
# for transient data
if pre_fire:
    for year in tqdm(years_transient):
        f = path+f'ELM_MOSART_CONUS.{run}.elm.h0.{year}-01-01-00000.nc'
        data = xr.open_dataset(f)
        for day in range(365):
            day_index = day+365*(year-years_transient[0])
            label = str(day+365*(year-years_transient[0]))

            # [1] for DOC injection flux, it's interpolated from ELM mesh to hillslope mesh
            #f_DOM, fdom = 1, '1'
            f_DOM, fdom = 0.01, '001'
            elm_flux_data_transient['DOC production latlon [molC m^-3 s^-1]'][label] = 0
            # theta = data['H2OSOI'].values[day, :10, :, :].mean(axis=0)
            # theta[np.isnan(theta)] = 0 
            # assert(len(np.unique(np.isnan(theta))) == 1)
            # for i, _key in enumerate(_keys):
            #     assert(len(np.unique(np.isnan(elm_data['DOC production latlon [molC m^-2 s^-1]'][label]))) == 1)
            #     elm_data['DOC production latlon [molC m^-2 s^-1]'][label] += data[_key].values[day, :, :]*f_DOM*theta/12#*area.mean()
            # elm_data['DOC production latlon [molC m^-3 s^-1]'][label] /= len(_keys)
            assert(len(np.unique(np.isnan(elm_flux_data_transient['DOC production latlon [molC m^-3 s^-1]'][label]))) == 1)
            elm_flux_data_transient['DOC production latlon [molC m^-3 s^-1]'][label] = data['HR'].values[day, :, :]*f_DOM/soil_thickness_median/12
            values = elm_flux_data_transient['DOC production latlon [molC m^-3 s^-1]'][label].flatten()
            interpolated_values = idw_interpolation(xy_target, xy_source, values)
            elm_flux_data_transient['DOC production [molC m^-3 s^-1]'][label] = interpolated_values.reshape(len(target_y_h5file), len(target_x_h5file))

            # [2] for NH4+ NO3- boundary concentration conditions
            #print(data['SMIN_NH4_vr'].values.shape) # (365, 15, 96, 144)
            values = data['H2OSOI'].values[day, :, hillslope_elm_lat_indices, hillslope_elm_lon_indices] #unit L/L
            elm_conc_data_transient['ELM water volume per bulk volume [L L^-1]'][day_index] = values.mean(axis=0)
            tmp_water_volume_soil_layers = values[:, 0:num_soil_layer_thickness_median].mean(axis=0)
            tmp_total_water_volume_soil_layers = sum(tmp_water_volume_soil_layers*soil_layer_thickness_array)/sum(soil_layer_thickness_array) #m3/m3
            
            values = data['SMIN_NH4_vr'].values[day, :, hillslope_elm_lat_indices, hillslope_elm_lon_indices]/14/1000 #unit mol/L
            elm_conc_data_transient['ELM NH4+ [mol L^-1]'][day_index] = values.mean(axis=0)
            values_soil = (values[:, 0:num_soil_layer_thickness_median]).mean(axis=0)
            elm_conc_data_transient['NH4+ bulk volume basis [molS L^-1]'][day_index] = (sum(values_soil*soil_layer_thickness_array)/sum(soil_layer_thickness_array))
            elm_conc_data_transient['NH4+ mol water basis [molS molH^-1]'][day_index] = (sum(values_soil*soil_layer_thickness_array)/sum(soil_layer_thickness_array))/tmp_total_water_volume_soil_layers*1000/rho_m
            
            values = data['SMIN_NO3_vr'].values[day, :, hillslope_elm_lat_indices, hillslope_elm_lon_indices]/14/1000 #unit mol/L
            elm_conc_data_transient['ELM NO3- [mol L^-1]'][day_index] = values.mean(axis=0)
            values_soil = (values[:, 0:num_soil_layer_thickness_median]).mean(axis=0)
            elm_conc_data_transient['NO3- bulk volume basis [molS L^-1]'][day_index] = (sum(values_soil*soil_layer_thickness_array)/sum(soil_layer_thickness_array))
            elm_conc_data_transient['NO3- mol water basis [molS molH^-1]'][day_index] = (sum(values_soil*soil_layer_thickness_array)/sum(soil_layer_thickness_array))/tmp_total_water_volume_soil_layers*1000/rho_m

            elm_conc_data_transient['time [d]'][day_index] = day_index

            # [3] Carbon pool concentrations
            varnames = ["LITR1C_vr", "LITR2C_vr", "LITR3C_vr",
            "SOIL1C_vr", "SOIL2C_vr", "SOIL3C_vr", "SOIL4C_vr"]
            for varname in varnames:
                values = data[varname].values[day, :, hillslope_elm_lat_indices, hillslope_elm_lon_indices]/12/1000 #unit mol/L
                elm_carbon_data_transient[f'ELM {varname} [mol L^-1]'][day_index] = values.mean(axis=0)
            
            # [4] DOC concentration, two options
            elm_conc_data_transient["DOC bulk volume basis v1 [molS L^-1]"][day_index] = elm_flux_data_transient["DOC production [molC m^-3 s^-1]"][label].mean()/1000# molC/L, if k_in_sec=1.0 s^-1
            elm_conc_data_transient["DOC mol water basis v1 [molS molH^-1]"][day_index] = elm_conc_data_transient["DOC bulk volume basis v1 [molS L^-1]"][day_index]/tmp_total_water_volume_soil_layers*1000/rho_m
            

In [ ]:
# [to do]
# if not pre_fire:
#     run = '2024-11-20-220109' # post-fire:'2024-08-27-150810'
#                               # fire: '2024-11-20-210107' no fire: '2024-11-20-220109'
#     path = f'/pscratch/sd/l/lizh142/elm/ELM_MOSART_CONUS.{run}/run/'
#     fl = sorted(glob.glob(path+f'ELM_MOSART_CONUS.{run}.elm.h0.*.nc'))
#     for i, f in enumerate(tqdm(fl)):
#         data = xr.open_dataset(f)
#         label = str(i)
#         f_DOM, fdom = 0.01, '001'
#         elm_data['DOC production latlon [molC m^-2 s^-1]'][label] = 0
#         theta = data['H2OSOI'].values[0, :10, :, :].mean(axis=0)
#         theta[np.isnan(theta)] = 0 
#         assert(len(np.unique(np.isnan(theta))) == 1)
#         for i, _key in enumerate(_keys):
#             assert(len(np.unique(np.isnan(elm_data['DOC production latlon [molC m^-2 s^-1]'][label]))) == 1)
#             elm_data['DOC production latlon [molC m^-2 s^-1]'][label] += data[_key].values[0, :, :]*f_DOM*theta/12#*area.mean()
#         elm_data['DOC production latlon [molC m^-2 s^-1]'][label] /= len(_keys)
#         values = elm_data['DOC production latlon [molC m^-2 s^-1]'][label].flatten()
#         interpolated_values = idw_interpolation(xy_target, xy_source, values)
#         elm_data['DOC production [molC m^-2 s^-1]'][label] = interpolated_values.reshape(120, 120)

#     elm_data["time [s]"] = (np.arange(len(fl)) + 15180)*86400

In [ ]:
# for spinup_raw
if pre_fire:
    for year in tqdm(years_spinup):
        f = path+f'ELM_MOSART_CONUS.{run}.elm.h0.{year}-01-01-00000.nc'
        data = xr.open_dataset(f)
        for day in range(365):
            day_index = day+365*(year-years_spinup[0])
            label = str(day+365*(year-years_spinup[0]))

            # [1] for DOC injection flux, it's interpolated from ELM mesh to hillslope mesh
            #f_DOM, fdom = 1, '1'
            f_DOM, fdom = 0.01, '001'
            elm_flux_data_spinup_raw['DOC production latlon [molC m^-3 s^-1]'][label] = 0
            # theta = data['H2OSOI'].values[day, :10, :, :].mean(axis=0)
            # theta[np.isnan(theta)] = 0 
            # assert(len(np.unique(np.isnan(theta))) == 1)
            # for i, _key in enumerate(_keys):
            #     assert(len(np.unique(np.isnan(elm_data['DOC production latlon [molC m^-2 s^-1]'][label]))) == 1)
            #     elm_data['DOC production latlon [molC m^-2 s^-1]'][label] += data[_key].values[day, :, :]*f_DOM*theta/12#*area.mean()
            # elm_data['DOC production latlon [molC m^-3 s^-1]'][label] /= len(_keys)
            assert(len(np.unique(np.isnan(elm_flux_data_spinup_raw['DOC production latlon [molC m^-3 s^-1]'][label]))) == 1)
            elm_flux_data_spinup_raw['DOC production latlon [molC m^-3 s^-1]'][label] = data['HR'].values[day, :, :]*f_DOM/soil_thickness_median/12
            values = elm_flux_data_spinup_raw['DOC production latlon [molC m^-3 s^-1]'][label].flatten()
            interpolated_values = idw_interpolation(xy_target, xy_source, values)
            elm_flux_data_spinup_raw['DOC production [molC m^-3 s^-1]'][label] = interpolated_values.reshape(len(target_y_h5file), len(target_x_h5file))

            # [2] for NH4+ NO3- boundary concentration conditions
            #print(data['SMIN_NH4_vr'].values.shape) # (365, 15, 96, 144)
            values = data['H2OSOI'].values[day, :, hillslope_elm_lat_indices, hillslope_elm_lon_indices] #unit L/L
            elm_conc_data_spinup_raw['ELM water volume per bulk volume [L L^-1]'][day_index] = values.mean(axis=0)
            tmp_water_volume_soil_layers = values[:, 0:num_soil_layer_thickness_median].mean(axis=0)
            tmp_total_water_volume_soil_layers = sum(tmp_water_volume_soil_layers*soil_layer_thickness_array)/sum(soil_layer_thickness_array)
            
            values = data['SMIN_NH4_vr'].values[day, :, hillslope_elm_lat_indices, hillslope_elm_lon_indices]/14/1000 #unit mol/L
            elm_conc_data_spinup_raw['ELM NH4+ [mol L^-1]'][day_index] = values.mean(axis=0)
            values_soil = (values[:, 0:num_soil_layer_thickness_median]).mean(axis=0)
            elm_conc_data_spinup_raw['NH4+ bulk volume basis [molS L^-1]'][day_index] = (sum(values_soil*soil_layer_thickness_array)/sum(soil_layer_thickness_array))
            elm_conc_data_spinup_raw['NH4+ mol water basis [molS molH^-1]'][day_index] = (sum(values_soil*soil_layer_thickness_array)/sum(soil_layer_thickness_array))/tmp_total_water_volume_soil_layers*1000/rho_m
            
            values = data['SMIN_NO3_vr'].values[day, :, hillslope_elm_lat_indices, hillslope_elm_lon_indices]/14/1000 #unit mol/L
            elm_conc_data_spinup_raw['ELM NO3- [mol L^-1]'][day_index] = values.mean(axis=0)
            values_soil = (values[:, 0:num_soil_layer_thickness_median]).mean(axis=0)
            elm_conc_data_spinup_raw['NO3- bulk volume basis [molS L^-1]'][day_index] = (sum(values_soil*soil_layer_thickness_array)/sum(soil_layer_thickness_array))
            elm_conc_data_spinup_raw['NO3- mol water basis [molS molH^-1]'][day_index] = (sum(values_soil*soil_layer_thickness_array)/sum(soil_layer_thickness_array))/tmp_total_water_volume_soil_layers*1000/rho_m

            elm_conc_data_spinup_raw['time [d]'][day_index] = day_index

            # [3] Carbon pool concentrations
            varnames = ["LITR1C_vr", "LITR2C_vr", "LITR3C_vr",
            "SOIL1C_vr", "SOIL2C_vr", "SOIL3C_vr", "SOIL4C_vr"]
            for varname in varnames:
                values = data[varname].values[day, :, hillslope_elm_lat_indices, hillslope_elm_lon_indices]/12/1000 #unit mol/L
                elm_carbon_data_spinup_raw[f'ELM {varname} [mol L^-1]'][day_index] = values.mean(axis=0)
                
            # [4] DOC concentration, two options
            elm_conc_data_spinup_raw["DOC bulk volume basis v1 [molS L^-1]"][day_index] = elm_flux_data_spinup_raw["DOC production [molC m^-3 s^-1]"][label].mean()/1000# molC/L, if k_in_sec=1.0 s^-1
            elm_conc_data_spinup_raw["DOC mol water basis v1 [molS molH^-1]"][day_index] = elm_conc_data_spinup_raw["DOC bulk volume basis v1 [molS L^-1]"][day_index]/tmp_total_water_volume_soil_layers*1000/rho_m
        

In [ ]:
soil_layer_thickness_array/sum(soil_layer_thickness_array)

## write hdf5 files

In [ ]:
# k ranges from 0.1~1.5 h^-1, to convert DOC flux to DOC concentration
k_in_sec = np.array([0.1, 1.5])/3600.0
#k, k_label = k_in_sec[0], '01'
k, k_label = k_in_sec[1], '15'

### for case cybernetic transient run

In [ ]:
output_folder = '/pscratch/sd/x/xiao284/elmbyhuilin/'
if pre_fire:
    filename = f'{site_name}_DOC_source_transient_{years_transient[0]}_{years_transient[-1]}_fdom{fdom}_{run}.h5'
    filepath = os.path.join(output_folder, filename)
    write_daymet_h5(filepath, elm_flux_data_transient)

    filename = f'{site_name}_CNbc_conc_transient_{years_transient[0]}_{years_transient[-1]}_fdom{fdom}_k{k_label}_{run}.h5'
    filepath = os.path.join(output_folder, filename)
    with h5.File(filepath, "w") as hdf:
        hdf.create_dataset("Time", data = elm_conc_data_transient["time [s]"])
        hdf.create_dataset("NH4+ bulk volume basis [molS L^-1]", data = elm_conc_data_transient["NH4+ bulk volume basis [molS L^-1]"])
        hdf.create_dataset("NO3- bulk volume basis [molS L^-1]", data = elm_conc_data_transient["NO3- bulk volume basis [molS L^-1]"])
        hdf.create_dataset("DOC bulk volume basis v1 [molS L^-1]", data = elm_conc_data_transient["DOC bulk volume basis v1 [molS L^-1]"]/k)
        hdf.create_dataset("NH4+ mol water basis [molS molH^-1]", data = elm_conc_data_transient["NH4+ mol water basis [molS molH^-1]"])
        hdf.create_dataset("NO3- mol water basis [molS molH^-1]", data = elm_conc_data_transient["NO3- mol water basis [molS molH^-1]"])
        hdf.create_dataset("DOC mol water basis v1 [molS molH^-1]", data = elm_conc_data_transient["DOC mol water basis v1 [molS molH^-1]"]/k)
    
# else:
#     filename = f'{site_name}_DOC_{years[0]}_{years[-1]}_fdom{fdom}_postfire_{run}.h5'
#     filepath = os.path.join(output_folder, filename)
#     write_daymet_h5(filepath, elm_data)

### for case cyclic spinup
1. caseflow steady state spinup run0
2. caseflow cyclic spinup run1
3. **casecybernetic cyclic spinup run1**
4. ~~casecybernetic transient~~

In [ ]:
elm_flux_data_spinup = {
    "DOC production [molC m^-3 s^-1]": {},
    "time [s]": (np.arange(nyears_cyclic_spinup*365))*86400,
    "x [m]": target_x_h5file,
    "y [m]": target_y_h5file,
}

typ = np.zeros((365, 
                elm_flux_data_spinup_raw['DOC production [molC m^-3 s^-1]']['0'].shape[0], 
                elm_flux_data_spinup_raw['DOC production [molC m^-3 s^-1]']['0'].shape[1]))

for day in trange(365):
    avg = 0
    for year in years_spinup:
        label = str(day+365*(year-years_spinup[0]))
        avg += elm_flux_data_spinup_raw['DOC production [molC m^-3 s^-1]'][label]
    typ[day, :, :] = avg/len(years_spinup)

for year in tqdm(range(nyears_cyclic_spinup)):
    for day in range(365):
        label = str(day+365*(year))
        elm_flux_data_spinup['DOC production [molC m^-3 s^-1]'][label] = typ[day, :, :]

output_folder = '/pscratch/sd/x/xiao284/elmbyhuilin/'
filename = f'{site_name}_DOC_source_spinup_{nyears_cyclic_spinup}yr_{years_spinup[0]}_{years_spinup[-1]}_fdom{fdom}.h5'
filepath = os.path.join(output_folder, filename)
write_daymet_h5(filepath, elm_flux_data_spinup)

In [ ]:
elm_conc_data_spinup = {
    "NH4+ bulk volume basis [molS L^-1]": np.zeros(nyears_cyclic_spinup*365),
    "NO3- bulk volume basis [molS L^-1]": np.zeros(nyears_cyclic_spinup*365),
    "NH4+ mol water basis [molS molH^-1]": np.zeros(nyears_cyclic_spinup*365),
    "NO3- mol water basis [molS molH^-1]": np.zeros(nyears_cyclic_spinup*365),
    "time [s]": (np.arange(nyears_cyclic_spinup*365))*86400,
    "DOC bulk volume basis v1 [molS L^-1]": np.zeros(nyears_cyclic_spinup*365),
    "DOC mol water basis v1 [molS molH^-1]": np.zeros(nyears_cyclic_spinup*365),
    "DOC bulk volume basis v2 [molS L^-1]": np.zeros(nyears_cyclic_spinup*365),
    "DOC mol water basis v2 [molS molH^-1]": np.zeros(nyears_cyclic_spinup*365)
}

variables_to_process = [key for key in elm_conc_data_spinup.keys() if key != "time [s]"]

# Calculate typical year for each variable
typical_year_data = {}
for var in variables_to_process:
    typical_year_data[var] = np.zeros(365)
    for day in trange(365):
        daily_values = []
        for year in years_spinup:
            day_index = day + 365 * (year - years_spinup[0])
            daily_values.append(elm_conc_data_spinup_raw[var][day_index])
        typical_year_data[var][day] = np.mean(daily_values)
    
    # Tile the typical year data
    elm_conc_data_spinup[var] = np.tile(typical_year_data[var], nyears_cyclic_spinup)

In [ ]:
daily_values

In [ ]:
output_folder = '/pscratch/sd/x/xiao284/elmbyhuilin/'
filename = f'{site_name}_CNbc_conc_spinup_{nyears_cyclic_spinup}yr_{years_spinup[0]}_{years_spinup[-1]}_fdom{fdom}_k{k_label}.h5'
filepath = os.path.join(output_folder, filename)

with h5.File(filepath, "w") as hdf:
    hdf.create_dataset("Time", data=elm_conc_data_spinup["time [s]"])    
    # Write all other datasets
    for key, hdfdata in elm_conc_data_spinup.items():
        if key != "time [s]":  # Skip time since we already wrote it
            if key.startswith("DOC"):
                # Scale DOC data by dividing by k
                hdf.create_dataset(key, data=hdfdata/k)
            else:
                # Write other datasets as is
                hdf.create_dataset(key, data=hdfdata)

## plot domain average over time

### plot DOC injection flux

In [ ]:
data2plot = np.zeros(len(years_transient)*365)
for year in years_transient:
    for day in range(365):
        day_index = day+365*(year-years_transient[0])
        label = str(day+365*(year-years_transient[0]))
        data2plot[day_index] = elm_flux_data_transient["DOC production [molC m^-3 s^-1]"][label].mean()

data2plot_transient = data2plot.copy()

In [ ]:
plt.plot(elm_flux_data_transient["time [s]"]/86400, 
         data2plot, 
         marker='.', markersize=3,
         linestyle='-', linewidth=0.5,
         color='k')

In [ ]:
data2plot = np.zeros(len(years_spinup)*365)
for year in years_spinup:
    for day in range(365):
        day_index = day+365*(year-years_spinup[0])
        label = str(day+365*(year-years_spinup[0]))
        data2plot[day_index] = elm_flux_data_spinup_raw["DOC production [molC m^-3 s^-1]"][label].mean()

data2plot_spinup_raw = data2plot.copy()

In [ ]:
plt.plot(elm_flux_data_spinup_raw["time [s]"]/86400, 
         data2plot, 
         marker='.', markersize=3,
         linestyle='-', linewidth=0.5,
         color='k')

In [ ]:
data2plot = np.zeros(nyears_cyclic_spinup*365)
for year in range(nyears_cyclic_spinup):
    for day in range(365):
        day_index = day+365*(year)
        label = str(day+365*(year))
        data2plot[day_index] = elm_flux_data_spinup["DOC production [molC m^-3 s^-1]"][label].mean()

data2plot_spinup = data2plot.copy()

In [ ]:
plt.plot(elm_flux_data_spinup["time [s]"]/86400, 
         data2plot, 
         marker='.', markersize=3,
         linestyle='-', linewidth=0.5,
         color='k')

### plot concentrations

In [ ]:
# NH4
# elm_conc_data_spinup_raw['ELM NH4+ [mol L^-1]']
if len(years_spinup) < nyears_cyclic_spinup:
    time_range_plot = range(len(years_spinup)*365)
else:
    time_range_plot = range(len(nyears_cyclic_spinup)*365)

time_labels = elm_conc_data_spinup_raw["time [d]"]
conc_over_time = elm_conc_data_spinup_raw["ELM NH4+ [mol L^-1]"]
num_layers = conc_over_time.shape[1]
num_layers_plot = 10
cmap = plt.get_cmap("tab10")
tab10_colors = [cmap(i) for i in range(num_layers_plot)]

plt.figure(figsize=(10, 6))

# Iterate through each of the 15 layers
for i in range(num_layers_plot):
    # Plot the i-th column (which is the concentration for the i-th layer over all time steps)
    layer_concentration = conc_over_time[:, i]
    # Create a line plot for the layer
    # Use a log scale for y-axis if concentrations span many orders of magnitude
    plt.plot(time_labels[time_range_plot], layer_concentration[time_range_plot], 
             marker='o', markersize=3,
             linestyle='-', linewidth=0.5,
             color=tab10_colors[i], label=f'Layer {i+1}')

plt.plot(elm_conc_data_spinup["time [s]"][time_range_plot]/86400, 
         elm_conc_data_spinup["NH4+ bulk volume basis [molS L^-1]"][time_range_plot], 
         marker='.', markersize=3,
         linestyle='-', linewidth=0.5,
         color='k')

# Add plot labels and title
plt.xlabel("Time Step")
plt.ylabel("ELM NH4+ Concentration [mol L^-1]")
plt.title("Averaged $\\text{NH}_4^+$ Concentration Over Time for Each Layer")

# Add a legend to distinguish the layers
# Place the legend outside the plot to avoid clutter
plt.legend(title="Layers", bbox_to_anchor=(1.05, 1), loc="upper left")

# Adjust plot scale if needed (common for concentration data)
plt.yscale("log") # Use logarithmic scale for the Y-axis

#plt.grid(True, which="both", ls="--", linewidth=0.5)
plt.tight_layout(rect=[0, 0, 0.9, 1]) # Adjust layout for legend placement

plt.show()

In [ ]:
# NO3-
# elm_conc_data_spinup_raw['ELM NO3- [mol L^-1]']
if len(years_spinup) < nyears_cyclic_spinup:
    time_range_plot = range(len(years_spinup)*365)
else:
    time_range_plot = range(len(nyears_cyclic_spinup)*365)
    
time_labels = elm_conc_data_spinup_raw["time [d]"]
conc_over_time = elm_conc_data_spinup_raw["ELM NO3- [mol L^-1]"]
num_layers = conc_over_time.shape[1]
num_layers_plot = 10
cmap = plt.get_cmap("tab10")
tab10_colors = [cmap(i) for i in range(num_layers_plot)]

plt.figure(figsize=(10, 6), facecolor='white')

# Iterate through each of the 15 layers
for i in range(num_layers_plot):
    # Plot the i-th column (which is the concentration for the i-th layer over all time steps)
    layer_concentration = conc_over_time[:, i]
    # Create a line plot for the layer
    # Use a log scale for y-axis if concentrations span many orders of magnitude
    plt.plot(time_labels[time_range_plot], layer_concentration[time_range_plot], 
             marker='o', markersize=3,
             linestyle='-', linewidth=0.5,
             color=tab10_colors[i], label=f'Layer {i+1}')

plt.plot(elm_conc_data_spinup["time [s]"][time_range_plot]/86400, 
         elm_conc_data_spinup["NO3- bulk volume basis [molS L^-1]"][time_range_plot], 
         marker='.', markersize=3,
         linestyle='-', linewidth=0.5,
         color='k')

# Add plot labels and title
plt.xlabel("Time Step")
plt.ylabel("ELM NO3- Concentration [mol L^-1]")
plt.title("Averaged $\\text{NO}_3^-$ Concentration Over Time for Each Layer")

# Add a legend to distinguish the layers
# Place the legend outside the plot to avoid clutter
plt.legend(title="Layers", bbox_to_anchor=(1.05, 1), loc="upper left")

# Adjust plot scale if needed (common for concentration data)
plt.yscale("log") # Use logarithmic scale for the Y-axis
plt.ylim(1e-18, 1e-3)

#plt.grid(True, which="both", ls="--", linewidth=0.5)
plt.tight_layout(rect=[0, 0, 0.9, 1]) # Adjust layout for legend placement

plt.show()

In [ ]:
# plot concentration in molS/molH

if len(years_spinup) < nyears_cyclic_spinup:
    time_range_plot = range(len(years_spinup)*365)
else:
    time_range_plot = range(len(nyears_cyclic_spinup)*365)
    
time_labels = elm_conc_data_spinup_raw["time [d]"]
conc_over_time = elm_conc_data_spinup_raw["ELM water volume per bulk volume [L L^-1]"]
num_layers = conc_over_time.shape[1]
num_layers_plot = 10
cmap = plt.get_cmap("tab10")
tab10_colors = [cmap(i) for i in range(num_layers_plot)]

plt.figure(figsize=(10, 6), facecolor='white')

# Iterate through each of the 15 layers
for i in range(num_layers_plot):
    # Plot the i-th column (which is the concentration for the i-th layer over all time steps)
    layer_concentration = conc_over_time[:, i]
    # Create a line plot for the layer
    # Use a log scale for y-axis if concentrations span many orders of magnitude
    plt.plot(time_labels[time_range_plot], layer_concentration[time_range_plot], 
             marker='o', markersize=3,
             linestyle='-', linewidth=0.5,
             color=tab10_colors[i], label=f'Layer {i+1}')

# plt.plot(elm_conc_data_spinup["time [s]"][time_range_plot]/86400, 
#          elm_conc_data_spinup["NO3- bulk volume basis [molS L^-1]"][time_range_plot], 
#          marker='.', linestyle='-', color='k')

# Add plot labels and title
plt.xlabel("Time Step")
plt.ylabel("soil moisture [L L^-1]")
plt.title("Averaged soil moisture Over Time for Each Layer")

# Add a legend to distinguish the layers
# Place the legend outside the plot to avoid clutter
plt.legend(title="Layers", bbox_to_anchor=(1.05, 1), loc="upper left")

# Adjust plot scale if needed (common for concentration data)
#plt.yscale("log") # Use logarithmic scale for the Y-axis
#plt.ylim(1e-18, 1e-3)

#plt.grid(True, which="both", ls="--", linewidth=0.5)
plt.tight_layout(rect=[0, 0, 0.9, 1]) # Adjust layout for legend placement
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))

plt.plot(elm_conc_data_spinup["time [s]"][time_range_plot]/86400, 
         elm_conc_data_spinup["NH4+ bulk volume basis [molS L^-1]"][time_range_plot], 
         #marker='.', markersize=3,
         linestyle='-', linewidth=1,
         color='k',
         label='[NH4+] - bulk volume basis')

plt.plot(elm_conc_data_spinup["time [s]"][time_range_plot]/86400, 
         elm_conc_data_spinup["NO3- bulk volume basis [molS L^-1]"][time_range_plot], 
         #marker='.', markersize=3,
         linestyle='-', linewidth=1,
         color='grey',
         label='[NO3-] - bulk volume basis')

plt.plot(elm_conc_data_spinup["time [s]"][time_range_plot]/86400, 
         elm_conc_data_spinup["NH4+ mol water basis [molS molH^-1]"][time_range_plot]*rho_m/1000, 
         marker='x', markersize=3,
         linestyle=':', linewidth=0.5,
         color='k',
         label='[NH4+] - water volume basis')

plt.plot(elm_conc_data_spinup["time [s]"][time_range_plot]/86400, 
         elm_conc_data_spinup["NO3- mol water basis [molS molH^-1]"][time_range_plot]*rho_m/1000, 
         marker='x', markersize=3,
         linestyle=':', linewidth=0.5,
         color='grey',
         label='[NO3-] - water volume basis')

# Add plot labels and title
plt.xlabel("Time Step")
plt.ylabel("Concentration per bulk or water volume")
#plt.title("Concentrations")

# Add a legend to distinguish the layers
# Place the legend outside the plot to avoid clutter
plt.legend()

#plt.grid(True, which="both", ls="--", linewidth=0.5)
plt.tight_layout() # Adjust layout for legend placement

plt.show()

# [Explore] Carbon pool concentrations

## plot LIT and SOM pool concentrations
- [DOC] v1, estimated based on quasi-steady-state assumption
- [DOC] v2, estimated based on P_DOC/(P-ET)

In [ ]:
# k ranges from 0.1~1.5 h^-1, to convert DOC flux to DOC concentration
k_in_sec = np.array([0.1, 1.5])/3600.0

In [ ]:
varnames = ["LITR1C_vr", "LITR2C_vr", "LITR3C_vr",
            "SOIL1C_vr", "SOIL2C_vr", "SOIL3C_vr", "SOIL4C_vr"]

num_layers_plot = 10
cmap = plt.get_cmap("tab10")
tab10_colors = [cmap(i) for i in range(num_layers_plot)]

# Create a figure with 2x4 subplots
plt.figure(figsize=(14, 10), facecolor='white')

# Iterate through variables
for idx, varname in enumerate(varnames):
    # Create subplot
    plt.subplot(2, 4, idx + 1)
    
    # Get concentration data for this variable
    conc_over_time = elm_carbon_data_spinup_raw[f'ELM {varname} [mol L^-1]']
    time_labels = elm_conc_data_spinup_raw["time [d]"]
    
    # Iterate through layers
    for i in range(num_layers_plot):
        # Plot concentration for each layer
        layer_concentration = conc_over_time[:, i]
        plt.plot(time_labels[time_range_plot], layer_concentration[time_range_plot],
                 marker='o', markersize=3,
                 linestyle='-', linewidth=0.5,
                 color=tab10_colors[i], label=f'Layer {i+1}')
    
    # Add plot labels and title
    plt.xlabel("Time Step")
    plt.ylabel(f'ELM {varname} Concentration [mol L^-1]')
    plt.title(f"[{varname}] Over Time")
    plt.yscale("log")

# Add legend to the first subplot
plt.subplot(2, 4, 1)
plt.legend(title="Layers", loc="upper left")

# Leave the last subplot empty
plt.subplot(2, 4, 8)
#plt.axis('off')
plt.plot(elm_flux_data_spinup_raw["time [s]"]/86400, 
         data2plot_spinup_raw / k_in_sec[0], 
         marker='.', markersize=3,
         linestyle='-', linewidth=0.5,
         color='r')
plt.plot(elm_flux_data_spinup_raw["time [s]"]/86400, 
         data2plot_spinup_raw / k_in_sec[1], 
         marker='.', markersize=3,
         linestyle='-', linewidth=0.5,
         color='b')
plt.xlabel("Time Step")
plt.title("[DOC] Over Time")
plt.yscale("log")

# Adjust layout
plt.tight_layout()
plt.show()

# [Optional] Various plots

## ELM Plot examples

### spatial plot of 1-layer of soil moisture
- plot soil moisture - top layer - with Naches/Oak Creek/hillslope

In [ ]:
# find ELM grid that overlap with the hillslope
# 
lon, lat = data.coords['lon'].values-360, data.coords['lat'].values
X, Y = np.meshgrid(lon, lat)

# Flatten X, Y into points
elm_mesh_points = np.column_stack([X.ravel(), Y.ravel()])
tree = cKDTree(elm_mesh_points)
line_coords = list(reproj_bnd_shply.exterior.coords)

# Find nearest cell for each point along the line
distances, indices_flat = tree.query(line_coords)
indices_flat = np.unique(indices_flat)

# Convert flat indices back to 2D indices
hillslope_elm_lat_indices = indices_flat // X.shape[1]
hillslope_elm_lon_indices = indices_flat % X.shape[1]

print(X.shape)
print([len(lat), len(lon)])
print(hillslope_elm_lat_indices)
print(hillslope_elm_lon_indices)

In [ ]:
fig, [ax1, ax2, ax3] = plt.subplots(1, 3, figsize=(12,4), subplot_kw={'projection': cartopy.crs.LambertConformal()})
i=0 # 1st layer

# raw
EasyMap("10m", ax=ax1, crs=cartopy.crs.LambertConformal()).STATES().OCEAN().LAND().COUNTIES().LAKES()
ax1.set_extent([lon.min(), lon.max(), lat.min(), lat.max()], crs=cartopy.crs.PlateCarree())
ax1.add_feature(cartopy.feature.RIVERS)
art = ax1.pcolormesh(X, Y, data['H2OSOI'].values[60, i, :, :], cmap='Spectral_r', vmin=0, vmax=1, transform=cartopy.crs.PlateCarree())
fig.colorbar(art, label='[mm3/mm3]', shrink=0.5, extend='max', orientation='vertical', pad=0.02)
ax1.set_title(f'layer {i+1}: {str(np.around(data.levgrnd.values[i], 3))} m')
# add specified boundaries
ax1.plot(naches_wbd[:,0]+360, naches_wbd[:,1], 'k--', lw=2, transform=pc)
ax1.plot(oakcreek_wbd[:,0]+360, oakcreek_wbd[:,1], 'k', lw=2, transform=pc)
watershed_workflow.plot.shply(reproj_bnd_shply, crs_wgs84, ax=ax1, color='r')

# zoom in level1
lon_min_zoom = np.min(oakcreek_wbd[:, 0])+360
lon_max_zoom = np.max(oakcreek_wbd[:, 0])+360
lat_min_zoom = np.min(oakcreek_wbd[:, 1])
lat_max_zoom = np.max(oakcreek_wbd[:, 1])

EasyMap("10m", ax=ax2, crs=cartopy.crs.LambertConformal()).STATES().OCEAN().LAND().COUNTIES().LAKES()
ax2.add_feature(cartopy.feature.RIVERS)
art = ax2.pcolormesh(X, Y, data['H2OSOI'].values[60, i, :, :], cmap='Spectral_r', vmin=0, vmax=1, transform=cartopy.crs.PlateCarree())
fig.colorbar(art, label='[mm3/mm3]', shrink=0.5, extend='max', orientation='vertical', pad=0.02)
ax2.set_title(f'layer {i+1}: {str(np.around(data.levgrnd.values[i], 3))} m')
# add specified boundaries
#ax2.plot(naches_wbd[:,0]+360, naches_wbd[:,1], 'k--', lw=2, transform=pc)
ax2.plot(oakcreek_wbd[:,0]+360, oakcreek_wbd[:,1], 'k', lw=2, transform=pc)
watershed_workflow.plot.shply(reproj_bnd_shply, crs_wgs84, ax=ax2, color='r')
ax2.set_extent([lon_min_zoom, lon_max_zoom, lat_min_zoom, lat_max_zoom], crs=cartopy.crs.PlateCarree())

# zoom in level2
lon_min_zoom, lat_min_zoom, lon_max_zoom, lat_max_zoom = reproj_bnd_shply.bounds
lon_min_zoom +=360
lon_max_zoom +=360
buffer = 0.005

#EasyMap("10m", ax=ax3, crs=cartopy.crs.LambertConformal()).STATES().OCEAN().LAND().COUNTIES().LAKES()
#ax3.add_feature(cartopy.feature.RIVERS)
art = ax3.pcolormesh(X, Y, data['H2OSOI'].values[60, i, :, :], cmap='Spectral_r', vmin=0, vmax=1, transform=cartopy.crs.PlateCarree())
fig.colorbar(art, label='[mm3/mm3]', shrink=0.5, extend='max', orientation='vertical', pad=0.02)
ax3.set_title(f'layer {i+1}: {str(np.around(data.levgrnd.values[i], 3))} m')
# add specified boundaries
#ax3.plot(naches_wbd[:,0]+360, naches_wbd[:,1], 'k--', lw=2, transform=pc)
ax3.plot(oakcreek_wbd[:,0]+360, oakcreek_wbd[:,1], 'k', lw=2, transform=pc)
watershed_workflow.plot.shply(reproj_bnd_shply, crs_wgs84, ax=ax3, color='r')
if True: # can make it True after found overlap ELM grids
    ax3.scatter(X[hillslope_elm_lat_indices, hillslope_elm_lon_indices], 
                Y[hillslope_elm_lat_indices, hillslope_elm_lon_indices], 
               s=50, c='blue', marker='x', linewidths=2,
               transform=cartopy.crs.PlateCarree(), 
               label='Nearest cells', zorder=10)
ax3.set_extent([lon_min_zoom - buffer, lon_max_zoom + buffer, lat_min_zoom - buffer, lat_max_zoom + buffer], crs=cartopy.crs.PlateCarree())
plt.show()

### spatial plot of multi-layer

In [ ]:
# plot soil moisture
lon, lat = data.coords['lon'].values-360, data.coords['lat'].values
X, Y = np.meshgrid(lon, lat)

fig, axs = plt.subplots(5, 3, figsize=(12,12), subplot_kw={'projection': cartopy.crs.LambertConformal()})
for i, ax in enumerate(axs.flat):
    EasyMap("10m", ax=ax, crs=cartopy.crs.LambertConformal()).STATES().OCEAN().LAND().COUNTIES().LAKES()
    ax.set_extent([lon.min(), lon.max(), lat.min(), lat.max()], crs=cartopy.crs.PlateCarree())
    ax.add_feature(cartopy.feature.RIVERS)
    art = ax.pcolormesh(X, Y, data['H2OSOI'].values[60, i, :, :], cmap='Spectral_r', vmin=0, vmax=1, transform=cartopy.crs.PlateCarree())
    fig.colorbar(art, label='[mm3/mm3]', shrink=0.5, extend='max', orientation='vertical', pad=0.02)
    ax.set_title(f'layer {i+1}: {str(np.around(data.levgrnd.values[i], 3))} m')
    # add specified boundaries
    ax.plot(naches_wbd[:,0]+360, naches_wbd[:,1], 'k--', lw=2, transform=pc)
    ax.plot(oakcreek_wbd[:,0]+360, oakcreek_wbd[:,1], 'k', lw=2, transform=pc)
    #watershed_workflow.plot.shply(reproj_bnd_shply, crs_wgs84, ax=ax, color='r')
plt.tight_layout()
plt.show()

In [ ]:
# plot soil moisture in a soil column
plt.plot(data['H2OSOI'].values[160, :, 0, 0], -np.arange(len(data['H2OSOI'].values[60, :, 0, 0])), '-x')
plt.show()

### spatial plot of ET and its components

In [ ]:
_keys = ['QVEGT', 'QSOIL', 'QVEGE']

In [ ]:
_names, _units = [], []
for _key in _keys:
    for i, key in enumerate(keys):
        if key == _key:
            _units.append(units[i])#.replace('s', 'd'))
            _names.append(names[i])
pd.DataFrame(data={'keys': _keys, 'names': _names, 'units': _units})

In [ ]:
# test plot 1 time slice
year = 2021
t = 0 # range(365)

f = path+f'ELM_MOSART_CONUS.{run}.elm.h0.{year}-01-01-00000.nc'
data = xr.open_dataset(f)
lon, lat = data.coords['lon'].values-360, data.coords['lat'].values
X, Y = np.meshgrid(lon, lat)

fig, axs = plt.subplots(2, 3, figsize=(15, 10), subplot_kw={'projection': cartopy.crs.LambertConformal()})
et = 0

for i, ax in enumerate(axs.flat):
    if i < 3:
        #layer1 - background with river
        EasyMap("10m", ax=ax, crs=cartopy.crs.LambertConformal()).STATES().OCEAN().LAND().COUNTIES().LAKES()
        ax.set_extent([lon.min(), lon.max(), lat.min(), lat.max()], crs=cartopy.crs.PlateCarree())
        ax.add_feature(cartopy.feature.RIVERS)
        et += data[_keys[i]].values[t]*86400
        #layer2 - elm var projected
        art = ax.pcolormesh(X, Y, data[_keys[i]].values[t]*86400, cmap='Spectral_r', transform=cartopy.crs.PlateCarree(), vmin=0, vmax=1)
        fig.colorbar(art, label=_units[0], shrink=0.5, extend='max', orientation='vertical', pad=0.02)
        ax.set_title(_keys[i]+'\n'+_names[i]+'\n['+_units[i]+']')
        # layer3 - add specified boundaries
        ax.plot(naches_wbd[:,0]+360, naches_wbd[:,1], 'k--', lw=2, transform=pc)
        ax.plot(oakcreek_wbd[:,0]+360, oakcreek_wbd[:,1], 'k', lw=2, transform=pc)
    elif i == 3:
        ax.axis('off')
        continue
    elif i == 4:
        EasyMap("10m", ax=ax, crs=cartopy.crs.LambertConformal()).STATES().OCEAN().LAND().COUNTIES().LAKES()
        ax.set_extent([lon.min(), lon.max(), lat.min(), lat.max()], crs=cartopy.crs.PlateCarree())
        ax.add_feature(cartopy.feature.RIVERS)
        art = ax.pcolormesh(X, Y, et, cmap='Spectral_r', transform=cartopy.crs.PlateCarree(), vmin=0, vmax=1)
        fig.colorbar(art, label=_units[0], shrink=0.5, extend='max', orientation='vertical', pad=0.02)
        ax.set_title(f'Total ET\n{year}, Day {t+1}', fontsize=24)
        ax.plot(naches_wbd[:,0]+360, naches_wbd[:,1], 'k', lw=2, transform=pc)
        ax.plot(oakcreek_wbd[:,0]+360, oakcreek_wbd[:,1], 'k', lw=2, transform=pc)
    elif i == 5:
        EasyMap("10m", ax=ax, crs=cartopy.crs.LambertConformal()).STATES().OCEAN().LAND().COUNTIES().LAKES()
        ax.set_extent([lon.min(), lon.max(), lat.min(), lat.max()], crs=cartopy.crs.PlateCarree())
        ax.add_feature(cartopy.feature.RIVERS)
        art = ax.pcolormesh(X, Y, data['EFLX_LH_TOT'].values[t]*0.0345, cmap='Spectral_r', transform=cartopy.crs.PlateCarree(), vmin=0, vmax=1)
        fig.colorbar(art, label=_units[0], shrink=0.5, extend='max', orientation='vertical', pad=0.02)
        ax.set_title(f'Total ET from LH\n{year}, Day {t+1}', fontsize=24)
        ax.plot(naches_wbd[:,0]+360, naches_wbd[:,1], 'k', lw=2, transform=pc)
        ax.plot(oakcreek_wbd[:,0]+360, oakcreek_wbd[:,1], 'k', lw=2, transform=pc)
plt.tight_layout()

In [ ]:
if generate_animation == True:
    # [note] for three years data, it tooks 2.5h to complete
    for year in years:
        f = path+f'ELM_MOSART_CONUS.{run}.elm.h0.{year}-01-01-00000.nc'
        data = xr.open_dataset(f)
        lon, lat = data.coords['lon'].values-360, data.coords['lat'].values
        X, Y = np.meshgrid(lon, lat)
        
        for t in tqdm(range(365)):
            fig, axs = plt.subplots(2, 3, figsize=(15, 10), subplot_kw={'projection': cartopy.crs.LambertConformal()})
            et = 0
            for i, ax in enumerate(axs.flat):
                if i < 3:
                    EasyMap("10m", ax=ax, crs=cartopy.crs.LambertConformal()).STATES().OCEAN().LAND().COUNTIES().LAKES()
                    ax.set_extent([lon.min(), lon.max(), lat.min(), lat.max()], crs=cartopy.crs.PlateCarree())
                    ax.add_feature(cartopy.feature.RIVERS)
                    et += data[_keys[i]].values[t]*86400
                    art = ax.pcolormesh(X, Y, data[_keys[i]].values[t]*86400, cmap='Spectral_r', transform=cartopy.crs.PlateCarree(), vmin=0, vmax=1)
                    fig.colorbar(art, label=_units[0], shrink=0.5, extend='max', orientation='vertical', pad=0.02)
                    ax.set_title(_keys[i]+'\n'+_names[i]+'\n['+_units[i]+']')
                    # add specified boundaries
                    ax.plot(naches_wbd[:,0]+360, naches_wbd[:,1], 'k--', lw=2, transform=pc)
                    #ax.plot(oakcreek_wbd[:,0]+360, oakcreek_wbd[:,1], 'k', lw=2, transform=pc)
                elif i == 3:
                    ax.axis('off')
                    continue
                elif i == 4:
                    EasyMap("10m", ax=ax, crs=cartopy.crs.LambertConformal()).STATES().OCEAN().LAND().COUNTIES().LAKES()
                    ax.set_extent([lon.min(), lon.max(), lat.min(), lat.max()], crs=cartopy.crs.PlateCarree())
                    ax.add_feature(cartopy.feature.RIVERS)
                    art = ax.pcolormesh(X, Y, et, cmap='Spectral_r', transform=cartopy.crs.PlateCarree(), vmin=0, vmax=1)
                    fig.colorbar(art, label=_units[0], shrink=0.5, extend='max', orientation='vertical', pad=0.02)
                    ax.set_title(f'Total ET\n{year}, Day {t+1}', fontsize=24)
                    ax.plot(naches_wbd[:,0]+360, naches_wbd[:,1], 'k', lw=2, transform=pc)
                    #ax.plot(oakcreek_wbd[:,0]+360, oakcreek_wbd[:,1], 'k', lw=2, transform=pc)
                elif i == 5:
                    EasyMap("10m", ax=ax, crs=cartopy.crs.LambertConformal()).STATES().OCEAN().LAND().COUNTIES().LAKES()
                    ax.set_extent([lon.min(), lon.max(), lat.min(), lat.max()], crs=cartopy.crs.PlateCarree())
                    ax.add_feature(cartopy.feature.RIVERS)
                    art = ax.pcolormesh(X, Y, data['EFLX_LH_TOT'].values[t]*0.0345, cmap='Spectral_r', transform=cartopy.crs.PlateCarree(), vmin=0, vmax=1)
                    fig.colorbar(art, label=_units[0], shrink=0.5, extend='max', orientation='vertical', pad=0.02)
                    ax.set_title(f'Total ET from LH\n{year}, Day {t+1}', fontsize=24)
                    ax.plot(naches_wbd[:,0]+360, naches_wbd[:,1], 'k', lw=2, transform=pc)
                    #ax.plot(oakcreek_wbd[:,0]+360, oakcreek_wbd[:,1], 'k', lw=2, transform=pc)
            plt.tight_layout()
            plt.savefig(f'./figs/{year}_{str(t).zfill(4)}.jpg')
            plt.close()
        

In [ ]:
# ffmpeg manual version
# ffmpeg -framerate 10 -i ./figs/2021_%04d.jpg -c:v gif 2021_ET.gif
# ffmpeg -framerate 10 -i ./figs/2022_%04d.jpg -c:v gif 2022_ET.gif
# ffmpeg -framerate 10 -i ./figs/2023_%04d.jpg -c:v gif 2023_ET.gif

import sys
import os

home_dir = os.path.expanduser("~")
my_utils_path = os.path.join(home_dir, 'my_utils')
if my_utils_path not in sys.path:
    sys.path.append(my_utils_path)

from viz import make_gif_ffmpeg

if generate_animation == True:
    make_gif_ffmpeg(image_folder='./figs', fps=10, input_jpg_fname='2021_%04d.jpg', output_gif_fname='2021_ET.gif')
    make_gif_ffmpeg(image_folder='./figs', fps=10, input_jpg_fname='2022_%04d.jpg', output_gif_fname='2022_ET.gif')
    make_gif_ffmpeg(image_folder='./figs', fps=10, input_jpg_fname='2023_%04d.jpg', output_gif_fname='2023_ET.gif')

### calculate spatial mean of 2mT, rain, snowmelt, soil water content, and _key data, for temporal plot

In [ ]:
tsa, rain, snowmelt, theta = np.zeros(len(years)*365), np.zeros(len(years)*365), np.zeros(len(years)*365), np.zeros(len(years)*365)
for year in tqdm(years):
    f = path+f'ELM_MOSART_CONUS.{run}.elm.h0.{year}-01-01-00000.nc'
    data = xr.open_dataset(f)
    for j in range(365):
        tsa[j+365*(year-years[0])] = data['TSA'].values[j, :, :].mean()-273.15 # K -> deg C
        rain[j+365*(year-years[0])] = data['RAIN'].values[j, :, :].mean()*86400 # mm/s -> mm/d
        snowmelt[j+365*(year-years[0])] = data['QSNOMELT'].values[j, :, :].mean()*86400 # mm/s -> mm/d
        theta[j+365*(year-years[0])] = data['H2OSOI'][j, :10, :, :].mean() # mm3/mm3

In [ ]:
_keys = ['CWDC_TO_LITR2C', 'CWDC_TO_LITR3C', 
         'LITR1C_TO_SOIL1C', 'LITR2C_TO_SOIL2C', 'LITR3C_TO_SOIL3C', 
         'SOIL1C_TO_SOIL2C', 'SOIL2C_TO_SOIL3C', 'SOIL3C_TO_SOIL4C',
         'HR', 'SMIN_NH4_vr', 'SMIN_NO3_vr']
         #, 'CWDC_HR', 'LITR1_HR', 'LITR2_HR', 'LITR3_HR',
         #'SOIL1_HR', 'SOIL2_HR', 'SOIL3_HR', 'SOIL4_HR']

In [ ]:
_names, _units = [], []
for _key in _keys:
    for i, key in enumerate(keys):
        if key == _key:
            _units.append(units[i])#.replace('/s', '/d'))#.replace('gC', 'mgC')
            _names.append(names[i])
pd.DataFrame(data={'keys': _keys, 'names': _names, 'units': _units})

In [ ]:
#[note] get spatial average
pd_spatial_avg = np.zeros((len(years)*365, len(_keys)))
for year in tqdm(years):
    f = path+f'ELM_MOSART_CONUS.{run}.elm.h0.{year}-01-01-00000.nc'
    data = xr.open_dataset(f)
    for j in range(365):
        for i in range(len(_keys)):
            pd_spatial_avg[j+365*(year-years[0]), i] = data[_keys[i]].values[j, :, :].mean()#*86400#*1000 # gC/m^2/s -> gC/m^2/d

### compare decomp decompose cascade fluxes vs HR (heterotrophic respiration)

In [ ]:
num_plots = len(_keys)
ncols_plots = 4
nrows_plots = math.ceil(num_plots / ncols_plots)

fig, axes = plt.subplots(nrows_plots, ncols_plots, figsize=(6*ncols_plots, 4 * nrows_plots))
axes = axes.flatten()

for i in range(len(_keys)):
    axes[i].plot(pd_spatial_avg[:, i], label=_keys[i])
    axes[i].set_xlabel("Time")
    axes[i].set_ylabel(_keys[i])
    axes[i].set_title(f"{_keys[i]}")
    axes[i].tick_params(axis='x', rotation=30)
    axes[i].legend(fontsize='small')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(6*2, 4))

i=8
axes.plot(pd_spatial_avg[:, i], label=_keys[i])

tmp_hr_v1 = (pd_spatial_avg[:, 2] +
            pd_spatial_avg[:, 3] +
            pd_spatial_avg[:, 4] +
            pd_spatial_avg[:, 5] +
            pd_spatial_avg[:, 6] +
            pd_spatial_avg[:, 7])
axes.plot(tmp_hr_v1, label="sum of CF")

# r_l1_s1 = 0.39
# r_l2_s2 = 0.55
# r_l3_s3 = 0.29
# r_s1_s2 = 0.28
# r_s2_s3 = 0.46
# r_s3_s4 = 0.55
# r_s4    = 1.0
# tmp_hr_v2 = (pd_spatial_avg[:, 2] / (1-r_l1_s1)*r_l1_s1 +
#           pd_spatial_avg[:, 3] / (1-r_l2_s2)*r_l2_s2 +
#           pd_spatial_avg[:, 4] / (1-r_l3_s3)*r_l3_s3 +
#           pd_spatial_avg[:, 5] / (1-r_s1_s2)*r_s1_s2 +
#           pd_spatial_avg[:, 6] / (1-r_s2_s3)*r_s2_s3 +
#           pd_spatial_avg[:, 7] / (1-r_s3_s4)*r_s3_s4)
# axes.plot(tmp_hr_v2, label="incomplete HR")

axes.set_xlabel("Time")
axes.set_ylabel(_keys[i])
axes.set_title(f"{_keys[i]}")
axes.tick_params(axis='x', rotation=30)
axes.legend(fontsize='small')

plt.tight_layout()
plt.show()

### temporal plot of all _keys data

In [ ]:
# user configure
_keys = ['QVEGT', 'QSOIL', 'QVEGE', 'QINFL']
_names, _units = [], []
for _key in _keys:
    for i, key in enumerate(keys):
        if key == _key:
            _units.append(units[i])
            _names.append(names[i])
pd.DataFrame(data={'keys': _keys, 'names': _names, 'units': _units})

#[note] get spatial average
pd_spatial_avg = np.zeros((len(years)*365, len(_keys)))
for year in tqdm(years):
    f = path+f'ELM_MOSART_CONUS.{run}.elm.h0.{year}-01-01-00000.nc'
    data = xr.open_dataset(f)
    for j in range(365):
        for i in range(len(_keys)):
            pd_spatial_avg[j+365*(year-years[0]), i] = data[_keys[i]].values[j, :, :].mean()#*86400#*1000 # gC/m^2/s -> gC/m^2/d

In [ ]:
plt.figure(figsize=(20, 4))
for i in range(len(_keys)):
    plt.plot(pd_spatial_avg[:, i], label=_keys[i])
plt.xticks(np.arange(0, pd_spatial_avg.shape[0], 365), 
           np.arange(0, pd_spatial_avg.shape[0], 365)//365+years[0], rotation=45)
plt.legend(ncols=4, edgecolor='none', facecolor='none')
#plt.xlim(365*35, pd_spatial_avg.shape[0])
plt.grid(ls='--')
plt.ylabel(_units[0])

# plt.gca().twinx().plot(tsa, label='tsa', lw=3, color='navy', alpha=0.3)
# plt.ylim(-20, 40)
# plt.ylabel('[deg C]')

# plt.gca().twinx().plot(rain, label='rain', lw=3, color='navy', alpha=0.3)
# plt.ylim(-25, 110)
# plt.gca().invert_yaxis()
# plt.ylabel('[mm/d]')

# plt.gca().twinx().plot(snowmelt, label='snowmelt', lw=3, color='navy', alpha=0.3)
# plt.ylim(-25, 110)
# plt.gca().invert_yaxis()
# plt.ylabel('[mm/d]')

plt.grid()
plt.show()

In [ ]:
plt.figure(figsize=(20, 4))
plt.plot(theta)
plt.xticks(np.arange(0, pd_spatial_avg.shape[0], 365), 
           np.arange(0, pd_spatial_avg.shape[0], 365)//365+years[0], rotation=45)
#plt.xlim(365*35, pd_spatial_avg.shape[0])
plt.grid(ls='--')
plt.ylabel('[mm3/mm3]')

# plt.gca().twinx().plot(tsa, label='tsa', lw=3, color='navy', alpha=0.3)
# plt.ylim(-20, 40)
# plt.ylabel('[deg C]')

plt.gca().twinx().plot(rain, label='rain', lw=3, color='navy', alpha=0.3)
plt.ylim(-25, 110)
plt.gca().invert_yaxis()
plt.ylabel('[mm/d]')

# plt.gca().twinx().plot(snowmelt, label='snowmelt', lw=3, color='navy', alpha=0.3)
# plt.ylim(-25, 110)
# plt.gca().invert_yaxis()
# plt.ylabel('[mm/d]')

plt.grid()
plt.show()

### (optional) calculate typical year data

In [ ]:
t_typ, rain_typ, snowmelt_typ, theta_typ, pd_spatial_avg_typ = 0, 0, 0, 0, 0
for i in range(len(years)):
    t_typ += tsa[i*365:(i+1)*365]
    rain_typ += rain[i*365:(i+1)*365]
    snowmelt_typ += snowmelt[i*365:(i+1)*365]
    theta_typ += theta[i*365:(i+1)*365]
    pd_spatial_avg_typ += pd_spatial_avg[i*365:(i+1)*365, :]
t_typ /= len(years)
rain_typ /= len(years)
snowmelt_typ /= len(years)
theta_typ /= len(years)
pd_spatial_avg_typ /= len(years)

In [ ]:
np.mean(pd_spatial_avg_typ, axis=1).shape

In [ ]:
plt.figure(figsize=(10, 4))
for i in range(len(_keys)):
    plt.plot(pd_spatial_avg_typ[:, i]*86400, lw=2, label=_keys[i])
plt.xticks(np.linspace(0, 364, 13), ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec', 'Jan'])
plt.legend(ncols=4, edgecolor='none', facecolor='none')
plt.grid(ls='--')
plt.ylabel(_units[0].replace('/s', '/d'))
# plt.ylim(0, 160)

# plt.gca().twinx().plot(t_typ, label='tsa', lw=8, color='navy', alpha=0.3)
# plt.ylim(-15, 30)
# plt.ylabel('[deg C]')

# plt.gca().twinx().plot(rain_typ, label='rain', lw=3, color='navy', alpha=0.3)
# plt.ylim(-1, 6)
# plt.gca().invert_yaxis()
# plt.ylabel('[mm/d]')

# plt.gca().twinx().plot(snowmelt_typ, label='snowmelt', lw=3, color='navy', alpha=0.3)
# plt.ylim(-1, 6)
# plt.gca().invert_yaxis()
# plt.ylabel('[mm/d]')

plt.grid()
plt.show()

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(theta_typ, lw=2)
plt.xticks(np.linspace(0, 364, 13), ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec', 'Jan'])
plt.grid(ls='--')
plt.ylabel('[-]')
# plt.ylim(0, 160)

plt.gca().twinx().plot(t_typ, label='tsa', lw=8, color='navy', alpha=0.3)
plt.ylim(-8, 23)
# plt.gca().invert_yaxis()
plt.ylabel('[deg C]')

# plt.gca().twinx().plot(rain_typ, label='rain', lw=3, color='navy', alpha=0.3)
# plt.ylim(-1, 6)
# plt.gca().invert_yaxis()
# plt.ylabel('[mm/d]')

# plt.gca().twinx().plot(snowmelt_typ, label='snowmelt', lw=3, color='navy', alpha=0.3)
# plt.ylim(-1, 6)
# plt.gca().invert_yaxis()
# plt.ylabel('[mm/d]')

plt.grid()
plt.show()

In [ ]:
plt.figure(figsize=(10, 4))
total_flux = np.sum(pd_spatial_avg_typ, axis=1)
plt.plot(total_flux*86400, lw=2)
plt.xticks(np.linspace(0, 364, 13), ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec', 'Jan'])
plt.grid(ls='--')
plt.ylabel(_units[0].replace('/s', '/d'))
# plt.ylim(0, 160)

plt.gca().twinx().plot(t_typ, label='tsa', lw=8, color='navy', alpha=0.3)
plt.ylim(-8, 23)
# plt.gca().invert_yaxis()
plt.ylabel('[deg C]')

plt.grid()
plt.show()

### (optional) make daily images and GIFs

In [ ]:
# test plot 1 slice
year = 2021

f = path+f'ELM_MOSART_CONUS.{run}.elm.h0.{year}-01-01-00000.nc'
data = xr.open_dataset(f)
lon, lat = data.coords['lon'].values-360, data.coords['lat'].values
X, Y = np.meshgrid(lon, lat)

if True: #year == years[0]:
    keys, names, units = [], [], []
    for key in list(data.keys()):
        try:
            keys.append(key)
        except:
            keys.append('')
        try:
            names.append(data[key].long_name)
        except:
            names.append('')
        try:
            units.append(data[key].units)
        except:
            units.append('')

    _keys = ['CWDC_TO_LITR2C', 'CWDC_TO_LITR3C', '',
     'LITR1C_TO_SOIL1C', 'LITR2C_TO_SOIL2C', 'LITR3C_TO_SOIL3C', 
     'SOIL1C_TO_SOIL2C', 'SOIL2C_TO_SOIL3C', 'SOIL3C_TO_SOIL4C']
    _names, _units = [], []
    for j, _key in enumerate(_keys):
        if j == 2:
            _units.append('')
            _names.append('')
        for i, key in enumerate(keys):
            if key == _key:
                _units.append(units[i])#.replace('gC', 'mgC').replace('/s', '/d'))
                _names.append(names[i])

t=0 #range(365)
fig, axs = plt.subplots(3, 3, figsize=(15, 15), subplot_kw={'projection': cartopy.crs.LambertConformal()})
for i, ax in enumerate(axs.flat):
    if i == 2:
        ax.set_title(f'{year}\nDay {t+1}', fontsize=36)
        ax.axis('off')
        continue
    EasyMap("10m", ax=ax, crs=cartopy.crs.LambertConformal()).STATES().OCEAN().LAND().COUNTIES().LAKES()
    ax.set_extent([lon.min(), lon.max(), lat.min(), lat.max()], crs=cartopy.crs.PlateCarree())
    ax.add_feature(cartopy.feature.RIVERS)
    art = ax.pcolormesh(X, Y, data[_keys[i]].values[t]*1000*86400, cmap='Spectral_r', vmin=0, vmax=300, transform=cartopy.crs.PlateCarree())
    fig.colorbar(art, label='[mgC/m^2/d]', shrink=0.5, extend='max', orientation='vertical', pad=0.02)
    ax.set_title('\n\n\n'+_keys[i]+'\n'+_names[i])#+'\n['+_units[i]+']')
    ax.plot(naches_wbd[:,0]+360, naches_wbd[:,1], 'k', lw=2, transform=pc)
    ax.plot(oakcreek_wbd[:,0]+360, oakcreek_wbd[:,1], 'k', lw=2, transform=pc)
plt.tight_layout()
# plt.savefig(f'./figs/{year}_{str(t).zfill(4)}.jpg')
# plt.close()

In [ ]:
if generate_animation == True:
    for year in tqdm(years):
        f = path+f'ELM_MOSART_CONUS.{run}.elm.h0.{year}-01-01-00000.nc'
        data = xr.open_dataset(f)
        lon, lat = data.coords['lon'].values-360, data.coords['lat'].values
        X, Y = np.meshgrid(lon, lat)
        
        if year == years[0]:
            keys, names, units = [], [], []
            for key in list(data.keys()):
                try:
                    keys.append(key)
                except:
                    keys.append('')
                try:
                    names.append(data[key].long_name)
                except:
                    names.append('')
                try:
                    units.append(data[key].units)
                except:
                    units.append('')
    
            _keys = ['CWDC_TO_LITR2C', 'CWDC_TO_LITR3C', '',
             'LITR1C_TO_SOIL1C', 'LITR2C_TO_SOIL2C', 'LITR3C_TO_SOIL3C', 
             'SOIL1C_TO_SOIL2C', 'SOIL2C_TO_SOIL3C', 'SOIL3C_TO_SOIL4C']
            _names, _units = [], []
            for j, _key in enumerate(_keys):
                if j == 2:
                    _units.append('')
                    _names.append('')
                for i, key in enumerate(keys):
                    if key == _key:
                        _units.append(units[i])#.replace('gC', 'mgC').replace('/s', '/d'))
                        _names.append(names[i])
    
        for t in tqdm(range(365)):
            fig, axs = plt.subplots(3, 3, figsize=(15, 15), subplot_kw={'projection': cartopy.crs.LambertConformal()})
            for i, ax in enumerate(axs.flat):
                if i == 2:
                    ax.set_title(f'{year}\nDay {t+1}', fontsize=36)
                    ax.axis('off')
                    continue
                EasyMap("10m", ax=ax, crs=cartopy.crs.LambertConformal()).STATES().OCEAN().LAND().COUNTIES().LAKES()
                ax.set_extent([lon.min(), lon.max(), lat.min(), lat.max()], crs=cartopy.crs.PlateCarree())
                ax.add_feature(cartopy.feature.RIVERS)
                art = ax.pcolormesh(X, Y, data[_keys[i]].values[t]*1000*86400, cmap='Spectral_r', vmin=0, vmax=300, transform=cartopy.crs.PlateCarree())
                fig.colorbar(art, label='[mgC/m^2/d]', shrink=0.5, extend='max', orientation='vertical', pad=0.02)
                ax.set_title('\n\n\n'+_keys[i]+'\n'+_names[i])#+'\n['+_units[i]+']')
                ax.plot(naches_wbd[:,0]+360, naches_wbd[:,1], 'k', lw=2, transform=pc)
                #ax.plot(oakcreek_wbd[:,0]+360, oakcreek_wbd[:,1], 'k', lw=2, transform=pc)
            plt.tight_layout()
            plt.savefig(f'./figs_carbon/{year}_{str(t).zfill(4)}.jpg')
            plt.close()
        
        #make_gif(fps=10, gif_fname=f'{year}.gif')

In [ ]:
import sys
import os

home_dir = os.path.expanduser("~")
my_utils_path = os.path.join(home_dir, 'my_utils')
if my_utils_path not in sys.path:
    sys.path.append(my_utils_path)

from viz import make_gif_ffmpeg

if generate_animation == True:
    make_gif_ffmpeg(image_folder='./figs_carbon', fps=10, input_jpg_fname='2021_%04d.jpg', output_gif_fname='2021_carbon.gif')
    make_gif_ffmpeg(image_folder='./figs_carbon', fps=10, input_jpg_fname='2022_%04d.jpg', output_gif_fname='2022_carbon.gif')
    make_gif_ffmpeg(image_folder='./figs_carbon', fps=10, input_jpg_fname='2023_%04d.jpg', output_gif_fname='2023_carbon.gif')